# Entity Resolution Maturity Journey - Master Orchestrator

This notebook executes the full 15-phase pipeline end-to-end on Microsoft Fabric.
Each phase can also be run independently for testing and incremental processing.

## Medallion Architecture
- **Bronze** (Phases 1-3): Ingestion, validation, quality gates
- **Silver** (Phases 4-8): Standardization, enrichment, dedup, fuzzy matching, blocking
- **Gold** (Phases 9-15): ML matching, LLM, embeddings, golden records, stewardship, MDM distribution

## Usage
1. Attach this notebook to your Fabric Lakehouse
2. Update the config path below to point to your pipeline config
3. Run all cells for full pipeline execution, or run individual phases

In [ ]:
# %%configure
# {
#     "defaultLakehouse": {
#         "name": "er_lakehouse"
#     },
#     "conf": {
#         "spark.sql.shuffle.partitions": "200",
#         "spark.sql.adaptive.enabled": "true",
#         "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
#         "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
#     }
# }

In [ ]:
# Import utilities and configure logging
import sys
sys.path.insert(0, 'examples/src')

from utils.spark_session import get_or_create_spark_session
from utils.logging_config import configure_logging, get_logger
from utils.metrics import MetricsCollector

configure_logging(level="INFO")
logger = get_logger(__name__)
spark = get_or_create_spark_session("EntityResolution-Master")

logger.info("Spark session initialized")
print(f"Spark version: {spark.version}")

In [ ]:
# Load pipeline configuration
import yaml

with open("examples/config/pipeline-config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Override with environment-specific config if available
import os
env = os.environ.get("FABRIC_ENV", "dev")
env_config_path = f"examples/config/fabric-{env}.yaml"
try:
    with open(env_config_path, "r") as f:
        env_config = yaml.safe_load(f)
    config.update(env_config)
    logger.info(f"Applied environment config: {env_config_path}")
except FileNotFoundError:
    logger.info(f"No environment config found at {env_config_path}, using defaults")

workspace = config.get("environment", {}).get("fabric_workspace")
logger.info(f"Pipeline configured for workspace: {workspace}")

In [ ]:
# ---------------------------------------------------------------------------
# Act I: Foundation (Phases 1-5) - Getting Data Under Control
# ---------------------------------------------------------------------------

from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

# Define expected schema for customer entity
customer_schema = StructType([
    StructField("id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("created_date", DateType(), True),
    StructField("updated_at", TimestampType(), True),
])

# Phase 1: Ingestion
import phase_01_ingestion as p1
ingestion_config = config.get("ingestion", {}).get("sources", [{}])[0]
bronze_path = config.get("medallion", {}).get("bronze_path", "Tables/bronze/")

logger.info("=== Phase 1: Basic Data Ingestion ===")
df_bronze = p1.run(
    spark,
    source_config=ingestion_config,
    bronze_path=bronze_path,
    source_system="crm_salesforce",
    workspace=workspace,
)
print(f"Phase 1 complete: {df_bronze.count()} records ingested")

# Phase 2: Schema Validation
import phase_02_schema_validation as p2
logger.info("=== Phase 2: Schema Validation ===")
df_valid = p2.run(
    spark,
    df_bronze,
    expected_schema=customer_schema,
    bronze_path=bronze_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 2 complete: {df_valid.count()} valid records")

# Phase 3: Data Quality
import phase_03_data_quality as p3
silver_path = config.get("medallion", {}).get("silver_path", "Tables/silver/")
logger.info("=== Phase 3: Data Quality Rules ===")
df_quality = p3.run(
    spark,
    df_valid,
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
    gateway_threshold=config.get("data_quality", {}).get("gateway_threshold", 1.0),
)
print(f"Phase 3 complete: {df_quality.count()} records passed quality gate")

# Phase 4: Standardization
import phase_04_standardization as p4
logger.info("=== Phase 4: Standardization ===")
df_std = p4.run(
    spark,
    df_quality,
    entity_type="customer",
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 4 complete: {df_std.count()} records standardized")

# Phase 5: Enrichment
import phase_05_enrichment as p5
env_config = config.get("enrichment", {})
logger.info("=== Phase 5: Data Enrichment ===")
df_enriched = p5.run(
    spark,
    df_std,
    entity_type="customer",
    config=env_config,
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 5 complete: {df_enriched.count()} records enriched")

In [ ]:
# ---------------------------------------------------------------------------
# Act II: Matching Mastery (Phases 6-12) - Finding the Same Entity
# ---------------------------------------------------------------------------

# Phase 6: Exact Deduplication
import phase_06_exact_dedup as p6
dedup_config = config.get("exact_dedup", {})
logger.info("=== Phase 6: Exact Deduplication ===")
df_dedup = p6.run(
    spark,
    df_enriched,
    entity_type="customer",
    config=dedup_config,
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 6 complete: {df_dedup.count()} unique records")

# Phase 7: Fuzzy Matching
import phase_07_fuzzy_matching as p7
fuzzy_config = config.get("fuzzy_matching", {})
logger.info("=== Phase 7: Fuzzy Matching ===")
pairs_fuzzy = p7.run(
    spark,
    df_dedup,
    entity_type="customer",
    config=fuzzy_config,
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 7 complete: {pairs_fuzzy.count()} fuzzy candidate pairs")

# Phase 8: Record Blocking
import phase_08_record_blocking as p8
block_config = config.get("record_blocking", {})
logger.info("=== Phase 8: Record Blocking ===")
df_blocked = p8.run(
    spark,
    df_dedup,
    entity_type="customer",
    config=block_config,
    silver_path=silver_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 8 complete: {df_blocked.count()} records blocked")

In [ ]:
# ---------------------------------------------------------------------------
# Act II continued: AI Matching (Phases 9-12)
# ---------------------------------------------------------------------------

gold_path = config.get("medallion", {}).get("gold_path", "Tables/gold/")

# Phase 9: Feature Engineering
# Requires candidate pairs from Phase 7
import phase_09_feature_engineering as p9
feat_config = config.get("feature_engineering", {})
logger.info("=== Phase 9: Feature Engineering ===")
if pairs_fuzzy.count() > 0:
    features_df = p9.run(
        spark,
        pairs_fuzzy,
        entity_type="customer",
        config=feat_config,
        gold_path=gold_path,
        table_name="customer",
        workspace=workspace,
    )
    print(f"Phase 9 complete: {features_df.count()} feature vectors")
else:
    logger.warning("No candidate pairs available; skipping Phase 9")
    features_df = None

# Phase 10: Probabilistic Matching
import phase_10_probabilistic_matching as p10
ml_config = config.get("probabilistic_matching", {})
logger.info("=== Phase 10: Probabilistic Matching ===")
if features_df is not None and features_df.count() > 0:
    ml_result = p10.run(
        spark,
        features_df,
        entity_type="customer",
        config=ml_config,
        gold_path=gold_path,
        table_name="customer",
        workspace=workspace,
    )
    scored_pairs = ml_result["scored_pairs"]
    print(f"Phase 10 complete: AUC={ml_result['metrics']['auc_roc']:.4f}")
else:
    logger.warning("No feature vectors available; skipping Phase 10")
    scored_pairs = None

# Phase 11: LLM Semantic Matching
import phase_11_semantic_matching_llm as p11
llm_config = config.get("llm_semantic", {})
logger.info("=== Phase 11: LLM Semantic Matching ===")
if scored_pairs is not None and scored_pairs.count() > 0:
    llm_result = p11.run(
        spark,
        scored_pairs,
        original_df=df_dedup,
        entity_type="customer",
        config=llm_config,
        gold_path=gold_path,
        table_name="customer",
        workspace=workspace,
    )
    print(f"Phase 11 complete: {llm_result.count()} pairs with LLM review")
    final_matches = llm_result
else:
    logger.warning("No scored pairs available; skipping Phase 11")
    final_matches = None

# Phase 12: Embedding-Based Matching
import phase_12_embedding_matching as p12
emb_config = config.get("embedding_matching", {})
logger.info("=== Phase 12: Embedding-Based Matching ===")
emb_result = p12.run(
    spark,
    df_dedup,
    entity_type="customer",
    config=emb_config,
    gold_path=gold_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 12 complete: {len(emb_result['match_pairs'])} embedding matches")

In [ ]:
# ---------------------------------------------------------------------------
# Act III: Operationalization (Phases 13-15) - Making It Real
# ---------------------------------------------------------------------------

# Phase 13: Golden Record Creation
import phase_13_golden_record as p13
gr_config = config.get("golden_record", {})
logger.info("=== Phase 13: Golden Record Creation ===")
golden_df = p13.run(
    spark,
    df_dedup,
    match_pairs=final_matches if final_matches is not None else None,
    entity_type="customer",
    config=gr_config,
    gold_path=gold_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 13 complete: {golden_df.count()} golden records")

# Phase 14: Data Stewardship
import phase_14_stewardship as p14
stew_config = config.get("stewardship", {})
logger.info("=== Phase 14: Data Stewardship ===")
stew_result = p14.run(
    spark,
    match_df=final_matches if final_matches is not None else None,
    entity_type="customer",
    config=stew_config,
    gold_path=gold_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 14 complete: {stew_result['metrics'].get('total_review_tasks', 0)} review tasks")

# Phase 15: MDM Distribution
import phase_15_mdm_distribution as p15
mdm_config = config.get("mdm_distribution", {})
logger.info("=== Phase 15: MDM Distribution ===")
mdm_result = p15.run(
    spark,
    golden_df=golden_df,
    entity_type="customer",
    config=mdm_config,
    gold_path=gold_path,
    table_name="customer",
    workspace=workspace,
)
print(f"Phase 15 complete: {len(mdm_result.get('export_paths', []))} export formats, "
      f"{len(mdm_result.get('events', []))} events")

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Summary
# ---------------------------------------------------------------------------

print("=" * 60)
print("ENTITY RESOLUTION MATURITY JOURNEY - PIPELINE COMPLETE")
print("=" * 60)
print(f"""
Act I: Foundation
  Phase  1 - Ingestion:         {df_bronze.count()} records
  Phase  2 - Schema Validation: {df_valid.count()} valid records
  Phase  3 - Data Quality:      {df_quality.count()} passed quality gate
  Phase  4 - Standardization:   {df_std.count()} standardized records
  Phase  5 - Enrichment:         {df_enriched.count()} enriched records

Act II: Matching Mastery
  Phase  6 - Exact Dedup:       {df_dedup.count()} unique records
  Phase  7 - Fuzzy Matching:    {pairs_fuzzy.count()} candidate pairs
  Phase  8 - Record Blocking:    {df_blocked.count()} records blocked
  Phase  9 - Feature Engineering: {features_df.count() if features_df else 0} feature vectors
  Phase 10 - ML Matching:        AUC={ml_result['metrics']['auc_roc']:.4f} if ml_result else 'N/A'
  Phase 11 - LLM Semantic:       {final_matches.count() if final_matches else 0} pairs with LLM
  Phase 12 - Embedding Matching: {len(emb_result['match_pairs'])} embedding matches

Act III: Operationalization
  Phase 13 - Golden Records:     {golden_df.count()} golden records
  Phase 14 - Data Stewardship:   {stew_result['metrics'].get('total_review_tasks', 0)} review tasks
  Phase 15 - MDM Distribution:   {len(mdm_result.get('export_paths', []))} export formats
""")

logger.info("Pipeline execution complete")